# Kapitola 4: Oddělování dat a instrukcí

- [Lekce](#lesson)
- [Cvičení](#exercises)
- [Příkladové hřiště](#example-playground)

## Nastavení

Spusťte následující buňku pro nastavení, abyste načetli svůj API klíč a vytvořili pomocnou funkci `get_completion`.

In [ ]:
```python
%pip install anthropic --quiet

# Importujte modul hints z balíčku utils
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Importujte vestavěnou knihovnu regulárních výrazů v Pythonu
import re
from anthropic import AnthropicBedrock

%store -r MODEL_NAME
%store -r AWS_REGION

client = AnthropicBedrock(aws_region=AWS_REGION)

def get_completion(prompt, system=''):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": prompt}
        ],
        system=system
    )
    return message.content[0].text
```

---

## Lekce

Často nechceme psát celé prompty, ale místo toho chceme **šablony promptů, které lze později upravit s dalšími vstupními daty před odesláním do Claude**. To může být užitečné, pokud chcete, aby Claude dělal pokaždé to samé, ale data, která Claude používá pro svůj úkol, mohou být pokaždé jiná.

Naštěstí to můžeme udělat poměrně snadno tím, že **oddělíme pevnou kostru promptu od proměnlivého vstupu uživatele a poté nahradíme vstup uživatele do promptu** před odesláním celého promptu do Claude.

Níže si krok za krokem projdeme, jak napsat šablonu promptu, kterou lze nahradit, a také jak nahradit vstup uživatele.

### Příklady

V tomto prvním příkladu žádáme Claude, aby fungoval jako generátor zvuků zvířat. Všimněte si, že celý prompt odeslaný Claudovi je pouze `PROMPT_TEMPLATE` nahrazený vstupem (v tomto případě "Cow"). Všimněte si, že slovo "Cow" nahrazuje zástupný symbol `ANIMAL` pomocí f-stringu, když vytiskneme celý prompt.

**Poznámka:** V praxi nemusíte svou zástupnou proměnnou nazývat nijak konkrétně. V tomto příkladu jsme ji nazvali `ANIMAL`, ale stejně tak bychom ji mohli nazvat `CREATURE` nebo `A` (i když je obecně dobré mít názvy proměnných specifické a relevantní, aby byla šablona promptu snadno pochopitelná i bez nahrazení, jen pro snadnou čitelnost uživatelem). Jen se ujistěte, že jakýkoli název své proměnné použijete, je to, co použijete pro f-string šablony promptu.

In [ ]:
```python
# Obsah proměnné
ANIMAL = "Cow"

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Řeknu vám jméno zvířete. Prosím, odpovězte zvukem, který toto zvíře vydává. {ANIMAL}"

# Vytisknout odpověď Claude
print("--------------------------- Kompletní prompt s nahrazením proměnných ---------------------------")
print(PROMPT)
print("\n------------------------------------- Odpověď Claude -------------------------------------")
print(get_completion(PROMPT))
```

Proč bychom chtěli oddělit a nahradit vstupy tímto způsobem? No, **prompt šablony zjednodušují opakující se úkoly**. Řekněme, že vytvoříte strukturu promptu, která vyzývá třetí strany k odeslání obsahu do promptu (v tomto případě zvíře, jehož zvuk chtějí generovat). Tito třetí uživatelé nemusí psát ani vidět celý prompt. Vše, co musí udělat, je vyplnit proměnné.

Tuto náhradu zde provádíme pomocí proměnných a f-řetězců, ale můžete to také udělat pomocí metody format().

**Poznámka:** Prompt šablony mohou mít tolik proměnných, kolik si přejete!

Při zavádění substitučních proměnných tímto způsobem je velmi důležité **ujistit se, že Claude ví, kde proměnné začínají a končí** (oproti instrukcím nebo popisům úkolů). Podívejme se na příklad, kde není žádné oddělení mezi instrukcemi a substituční proměnnou.

Pro naše lidské oči je velmi jasné, kde proměnná začíná a končí v níže uvedené šabloně promptu. Nicméně, v plně substituovaném promptu se toto vymezení stává nejasným.

In [ ]:
```python
# Obsah proměnné
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Vytisknout Claudeovu odpověď
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))
```

Zde si **Claude myslí, že "Yo Claude" je součástí e-mailu, který má přepsat**! Můžete to poznat, protože začíná svůj přepis s "Dear Claude". Pro lidské oko je to jasné, zejména v šabloně promptu, kde e-mail začíná a končí, ale po substituci se to v promptu stává mnohem méně jasným.

Jak to vyřešíme? **Obalte vstup do XML tagů**! Udělali jsme to níže a jak můžete vidět, v výstupu už není "Dear Claude".

[XML tagy](https://docs.anthropic.com/claude/docs/use-xml-tags) jsou tagy s úhlovými závorkami jako `<tag></tag>`. Přicházejí v párech a skládají se z otevíracího tagu, jako je `<tag>`, a uzavíracího tagu označeného `/`, jako je `</tag>`. XML tagy se používají k obalení obsahu, takto: `<tag>content</tag>`.

**Poznámka:** I když Claude dokáže rozpoznat a pracovat s širokou škálou oddělovačů a delimiterů, doporučujeme, abyste **používali konkrétně XML tagy jako oddělovače** pro Claude, protože Claude byl speciálně vyškolen k rozpoznávání XML tagů jako mechanismu organizace promptů. Mimo volání funkcí **neexistují žádné speciální XML tagy, na které by byl Claude vyškolen a které byste měli používat k maximálnímu zvýšení výkonu**. Tímto způsobem jsme záměrně udělali Claude velmi přizpůsobitelným a flexibilním.

In [ ]:
```python
# Obsah proměnné
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Vytisknout odpověď od Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))
```

Podívejme se na další příklad, jak nám mohou pomoci XML značky.

V následujícím promptu **Claude nesprávně interpretuje, která část promptu je instrukce a která je vstup**. Nesprávně považuje `Each is about an animal, like rabbits` za součást seznamu kvůli formátování, i když uživatel (ten, kdo vyplňuje proměnnou `SENTENCES`) to pravděpodobně nechtěl.

In [ ]:
```python
# Obsah proměnné
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Vytisknout odpověď Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))
```

K vyřešení tohoto problému stačí **obklopit vstupní věty uživatele XML značkami**. Tím se Claude ukáže, kde začínají a končí vstupní data, i přes zavádějící pomlčku před `Each is about an animal, like rabbits.`

In [ ]:
```python
# Obsah proměnné
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Šablona promptu s místem pro obsah proměnné
PROMPT = f""" Níže je seznam vět. Řekni mi druhou položku na seznamu.

- Každá je o zvířeti, jako jsou králíci.
<sentences>
{SENTENCES}
</sentences>"""

# Vytiskni odpověď od Claude
print("--------------------------- Kompletní prompt s náhradou proměnných ---------------------------")
print(PROMPT)
print("\n------------------------------------- Odpověď od Claude -------------------------------------")
print(get_completion(PROMPT))
```

**Poznámka:** V nesprávné verzi promptu "Každý je o zvířeti" jsme museli zahrnout pomlčku, aby Claude reagoval nesprávně tak, jak jsme chtěli pro tento příklad. Toto je důležitá lekce o promptování: **malé detaily jsou důležité**! Vždy stojí za to **zkontrolovat vaše prompty na překlepy a gramatické chyby**. Claude je citlivý na vzory (v jeho raných letech, před doladěním, to byl nástroj pro predikci surového textu) a je pravděpodobnější, že udělá chyby, když vy děláte chyby, je chytřejší, když zníte chytře, hloupější, když zníte hloupě, a tak dále.

Pokud byste chtěli experimentovat s prompty z lekce, aniž byste měnili jakýkoli obsah výše, přejděte až na konec poznámkového bloku lekce a navštivte [**Example Playground**](#example-playground).

---

## Cvičení
- [Cvičení 4.1 - Téma Haiku](#exercise-41---haiku-topic)
- [Cvičení 4.2 - Otázka o psovi s překlepy](#exercise-42---dog-question-with-typos)
- [Cvičení 4.3 - Otázka o psovi část 2](#exercise-42---dog-question-part-2)

### Cvičení 4.1 - Téma Haiku
Upravte `PROMPT` tak, aby to byla šablona, která přijme proměnnou nazvanou `TOPIC` a vygeneruje haiku o tomto tématu. Toto cvičení je určeno pouze k otestování vašeho porozumění struktuře šablonování proměnných s f-řetězci.

In [ ]:
```python
# Obsah proměnné
TOPIC = "Pigs"

# Šablona promptu s místem pro obsah proměnné
PROMPT = f""

# Získání odpovědi od Claude
response = get_completion(PROMPT)

# Funkce pro hodnocení správnosti cvičení
def grade_exercise(text):
    return bool(re.search("pigs", text.lower()) and re.search("haiku", text.lower()))

# Tisk odpovědi od Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("Toto cvičení bylo správně vyřešeno:", grade_exercise(response))
```

❓ Pokud chcete nápovědu, spusťte buňku níže!

In [ ]:
print(hints.exercise_4_1_hint)

### Cvičení 4.2 - Otázka o psovi s překlepy
Opravte `PROMPT` přidáním XML tagů, aby Claude vytvořil správnou odpověď.

Snažte se neměnit nic jiného na promptu. Chaotické a chybné psaní je záměrné, abyste viděli, jak Claude reaguje na takové chyby.

In [ ]:
```python
# Obsah proměnné
QUESTION = "ar cn brown?"

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Získání odpovědi od Claude
response = get_completion(PROMPT)

# Funkce pro hodnocení správnosti cvičení
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Výpis odpovědi od Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("Toto cvičení bylo správně vyřešeno:", grade_exercise(response))
```

❓ Pokud chcete nápovědu, spusťte buňku níže!

In [ ]:
print(hints.exercise_4_2_hint)

### Cvičení 4.3 - Otázka o psovi, část 2
Opravte `PROMPT` **BEZ** přidání XML značek. Místo toho odstraňte pouze jedno nebo dvě slova z promptu.

Stejně jako u výše uvedených cvičení se snažte neměnit nic jiného na promptu. To vám ukáže, jaký druh jazyka Claude dokáže analyzovat a porozumět mu.

In [ ]:
```python
# Obsah proměnné
QUESTION = "ar cn brown?"

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Získání odpovědi od Claude
response = get_completion(PROMPT)

# Funkce pro hodnocení správnosti cvičení
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Výpis odpovědi od Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("Toto cvičení bylo správně vyřešeno:", grade_exercise(response))
```

❓ Pokud chcete nápovědu, spusťte buňku níže!

In [ ]:
print(hints.exercise_4_3_hint)

### Gratulujeme!

Pokud jste vyřešili všechny úkoly až do tohoto bodu, jste připraveni přejít k další kapitole. Šťastné promptování!

---

## Příklad hřiště

Toto je prostor, kde můžete volně experimentovat s příklady promptů uvedenými v této lekci a upravovat prompty, abyste viděli, jak to může ovlivnit odpovědi Claude.

In [ ]:
```python
# Obsah proměnné
ANIMAL = "Cow"

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Řeknu vám jméno zvířete. Prosím, odpovězte zvukem, který toto zvíře vydává. {ANIMAL}"

# Vytisknout odpověď Claude
print("--------------------------- Kompletní prompt s nahrazením proměnných ---------------------------")
print(PROMPT)
print("\n------------------------------------- Odpověď Claude -------------------------------------")
print(get_completion(PROMPT))
```

In [ ]:
```python
# Obsah proměnné
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Vytisknout Claudeovu odpověď
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))
```

In [ ]:
```python
# Obsah proměnné
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Vytisknout Claudeovu odpověď
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))
```

In [ ]:
```python
# Obsah proměnné
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Šablona promptu s místem pro obsah proměnné
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Vytisknout odpověď Claude
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))
```

In [ ]:
```python
# Obsah proměnné
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Šablona promptu s místem pro obsah proměnné
PROMPT = f""" Níže je seznam vět. Řekni mi druhou položku na seznamu.

- Každá je o zvířeti, jako jsou králíci.
<sentences>
{SENTENCES}
</sentences>"""

# Vytisknout odpověď od Claude
print("--------------------------- Kompletní prompt s náhradou proměnných ---------------------------")
print(PROMPT)
print("\n------------------------------------- Odpověď od Claude -------------------------------------")
print(get_completion(PROMPT))
```